# Notebook 3 - Market Reaction Analysis
### Not Yet Priced In: Post-Event Return Windows

This notebook:
1. Joins filings with stock prices
2. Calculates 1-day, 3-day, and 5-day post-filing returns
3. Computes market-adjusted returns vs S&P 500
4. Classifies each event: Immediate / Delayed / Gradual / No Reaction
5. Saves reaction scores back 

Prerequisite: Run Notebooks 1 and 2 first.

---

In [3]:
#Day 1

In [4]:
pip install yfinance

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 22.7 MB/s eta 0:00:00
  DEPRECATION: Building 'multitasking' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'multitasking'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15548 sha256=73262c92e1c2a04107e6e5301715b6c8e6d9fabeddbc7fe17499e876d2f0efb6
  Stored in directory: /Users/parulchaudhary/Library/Caches/pip/wheels/1e/df/0f/e2bbb22d689b30c681feb5410ab64a2523437b34c8ecfc6476
Successfully built multitasking
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Uninstalli

In [2]:
import pandas as pd
import yfinance as yf
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
import os
print(os.getcwd())

/Users/parulchaudhary/Desktop/SEM 2/Big Data/not-yet-priced-in/notebooks


In [15]:
# Load the filings data
df = pd.read_excel('/Users/parulchaudhary/Desktop/SEM 2/Predictive/Group Project/filings_raw.xlsx')
df['filed_date'] = pd.to_datetime(df['filed_date'])

print(f"Total filings: {len(df)}")
print(f"Companies: {df['ticker'].nunique()}")
print(f"Date range: {df['filed_date'].min().date()} to {df['filed_date'].max().date()}")
df.head(20)

Total filings: 474
Companies: 42
Date range: 2023-01-03 to 2023-12-29


,ticker,cik,accession_no,filed_date,form_type,company_name,filing_url,local_text_path,fetch_ok
0,AAPL,320193,0000320193-23-000104,2023-11-02,8-K,Apple Inc.,https://www.sec.gov/Archives/edgar/data/320193...,../data/filings_text/AAPL_0000320193_23_000104...,True
1,AAPL,320193,0000320193-23-000075,2023-08-03,8-K,Apple Inc.,https://www.sec.gov/Archives/edgar/data/320193...,../data/filings_text/AAPL_0000320193_23_000075...,True
2,AAPL,320193,0001140361-23-023909,2023-05-10,8-K,Apple Inc.,https://www.sec.gov/Archives/edgar/data/320193...,../data/filings_text/AAPL_0001140361_23_023909...,True
3,AAPL,320193,0000320193-23-000063,2023-05-04,8-K,Apple Inc.,https://www.sec.gov/Archives/edgar/data/320193...,../data/filings_text/AAPL_0000320193_23_000063...,True
4,AAPL,320193,0001140361-23-011192,2023-03-10,8-K,Apple Inc.,https://www.sec.gov/Archives/edgar/data/320193...,../data/filings_text/AAPL_0001140361_23_011192...,True
5,AAPL,320193,0000320193-23-000005,2023-02-02,8-K,Apple Inc.,https://www.sec.gov/Archives/edgar/data/320193...,../data/filings_text/AAPL_0000320193_23_000005...,True
6,MSFT,789019,0001193125-23-291720,2023-12-08,8-K,MICROSOFT CORP,https://www.sec.gov/Archives/edgar/data/789019...,../data/filings_text/MSFT_0001193125_23_291720...,True
7,MSFT,789019,0001193125-23-271376,2023-11-06,8-K,MICROSOFT CORP,https://www.sec.gov/Archives/edgar/data/789019...,../data/filings_text/MSFT_0001193125_23_271376...,True
8,MSFT,789019,0001193125-23-265616,2023-10-30,8-K,MICROSOFT CORP,https://www.sec.gov/Archives/edgar/data/789019...,../data/filings_text/MSFT_0001193125_23_265616...,True
9,MSFT,789019,0000950170-23-054848,2023-10-24,8-K,MICROSOFT CORP,https://www.sec.gov/Archives/edgar/data/789019...,../data/filings_text/MSFT_0000950170_23_054848...,True


In [26]:
# Fetch stock prices for all companies + S&P 500
tickers = df['ticker'].unique().tolist()
tickers.append('^GSPC')  # S&P 500

print(f"Downloading prices for {len(tickers)} tickers...")
prices = yf.download(tickers, start='2023-01-01', end='2023-12-31', auto_adjust=True)['Close']

print(f"Done! Price data shape: {prices.shape}")
prices.head(20)

[**********************53%                       ]  23 of 43 completed

[*********************100%***********************]  43 of 43 completed

Done! Price data shape: (250, 43)


Ticker,AAPL,ABBV,ADBE,AMZN,AXP,BA,CAT,CMCSA,COP,COST,...,SBUX,SLB,T,TSLA,UNH,UNP,VZ,WMT,XOM,^GSPC
Date,,,,,,,,,,,,,,,,,,,,,
2023-01-03,123.096031,143.135162,336.920013,85.820000,140.973785,195.389999,226.435379,29.830660,101.792809,434.452820,...,93.157585,47.490376,15.481896,108.099998,487.192291,192.466370,31.746319,46.052876,95.434196,3824.139893
2023-01-04,124.365677,144.289917,341.410004,85.139999,144.250946,203.639999,228.786163,30.686077,101.990768,437.596527,...,96.511353,47.674809,15.812354,113.639999,473.909668,194.005524,32.545513,46.104183,95.711960,3852.969971
2023-01-05,123.046806,144.113602,328.440002,83.120003,140.798981,204.990005,227.762451,31.013153,105.077309,431.491150,...,96.483650,48.569283,15.870184,110.339996,460.251282,188.294006,32.996555,45.947041,97.853432,3808.100098
2023-01-06,127.574203,146.810959,332.750000,86.080002,144.395187,213.000000,235.895477,31.776316,106.391098,462.813751,...,98.571678,50.256809,16.134552,113.059998,460.288879,196.573807,33.384277,47.072708,99.036171,3895.080078
2023-01-09,128.095840,142.500488,341.980011,87.360001,144.616302,208.570007,233.772171,31.491184,105.608215,458.864838,...,96.770058,51.308044,15.956881,119.769997,460.345245,196.063873,33.250061,46.485813,97.190376,3892.090088
2023-01-10,128.666672,140.719925,338.700012,89.870003,146.154770,206.690002,237.459503,31.767937,106.256119,461.404755,...,97.934174,51.603142,16.317247,118.849998,456.531372,196.824173,33.716225,46.456963,98.641937,3919.250000
2023-01-11,131.383133,138.542618,342.929993,95.089996,148.000931,208.029999,237.668060,31.826639,106.400116,464.193970,...,98.211349,51.935108,16.266966,123.220001,463.482635,198.660019,33.097351,46.864246,99.788818,3969.610107
2023-01-12,131.304398,135.454819,344.540009,95.269997,148.962479,214.320007,241.781967,32.447239,108.127838,462.392090,...,97.795601,53.484310,16.266966,123.559998,465.615082,199.058716,33.603695,46.440922,101.446449,3983.169922
2023-01-13,132.633072,136.682816,344.380005,98.120003,149.770172,214.130005,244.995331,32.648518,109.306671,465.095001,...,99.070587,53.742508,16.384295,122.400002,459.884949,197.705032,33.643887,46.594852,101.383728,3999.090088


In [27]:
# Calculate daily returns for all tickers
returns = prices.pct_change()

print("Daily returns calculated!")
print(f"Shape: {returns.shape}")
returns.head()

Daily returns calculated!
Shape: (250, 43)


Ticker,AAPL,ABBV,ADBE,AMZN,AXP,BA,CAT,CMCSA,COP,COST,...,SBUX,SLB,T,TSLA,UNH,UNP,VZ,WMT,XOM,^GSPC
Date,,,,,,,,,,,,,,,,,,,,,
2023-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-04,0.010314,0.008068,0.013327,-0.007924,0.023247,0.042223,0.010382,0.028676,0.001945,0.007236,...,0.036001,0.003884,0.021345,0.051249,-0.027264,0.007997,0.025174,0.001114,0.002911,0.007539
2023-01-05,-0.010605,-0.001222,-0.037990,-0.023726,-0.023930,0.006629,-0.004475,0.010659,0.030263,-0.013952,...,-0.000287,0.018762,0.003657,-0.029039,-0.028821,-0.029440,0.013859,-0.003408,0.022374,-0.011646
2023-01-06,0.036794,0.018717,0.013123,0.035611,0.025541,0.039075,0.035708,0.024608,0.012503,0.072592,...,0.021641,0.034745,0.016658,0.024651,0.000082,0.043973,0.011750,0.024499,0.012087,0.022841
2023-01-09,0.004089,-0.029361,0.027739,0.014870,0.001531,-0.020798,-0.009001,-0.008973,-0.007359,-0.008532,...,-0.018277,0.020917,-0.011012,0.059349,0.000122,-0.002594,-0.004020,-0.012468,-0.018638,-0.000768


In [ ]:
# For each filing, calculate post-event returns
#adj_ret is teh s and p500 adjusted return for that filing - to undertsand teh true reaction to the 
#filing
results = []

for _, row in df.iterrows():
    ticker = row['ticker']
    filing_date = row['filed_date']
    
    # Find the next available trading day on or after filing date
    try:
        # Get prices for this ticker after filing date
        ticker_prices = prices[ticker]
        sp500_prices = prices['^GSPC']
        
        # Find trading days after filing
        future_prices = ticker_prices[ticker_prices.index >= filing_date]
        future_sp500 = sp500_prices[sp500_prices.index >= filing_date]
        
        if len(future_prices) < 21:
            continue
            
        # Base price (day of filing)
        base_price = future_prices.iloc[0]
        base_sp500 = future_sp500.iloc[0]
        
        # Calculate returns at each window
        ret_1d  = (future_prices.iloc[1] - base_price) / base_price
        ret_3d  = (future_prices.iloc[3] - base_price) / base_price
        ret_5d  = (future_prices.iloc[5] - base_price) / base_price
        ret_20d = (future_prices.iloc[20] - base_price) / base_price
        
        # S&P 500 returns for same windows
        sp_1d  = (future_sp500.iloc[1] - base_sp500) / base_sp500
        sp_3d  = (future_sp500.iloc[3] - base_sp500) / base_sp500
        sp_5d  = (future_sp500.iloc[5] - base_sp500) / base_sp500
        sp_20d = (future_sp500.iloc[20] - base_sp500) / base_sp500
        
        # Market adjusted returns
        adj_1d  = ret_1d  - sp_1d
        adj_2d  = ret_2d  - sp_2d
        adj_3d  = ret_3d  - sp_3d
        adj_4d  = ret_4d  - sp_4d
        adj_5d  = ret_5d  - sp_5d
        adj_6d  = ret_6d  - sp_6d
        
        adj_20d = ret_20d - sp_20d
        
        results.append({
            'ticker': ticker,
            'company_name': row['company_name'],
            'accession_no': row['accession_no'],
            'filed_date': filing_date,
            'ret_1d': round(ret_1d, 4),
            'ret_3d': round(ret_3d, 4),
            'ret_5d': round(ret_5d, 4),
            'ret_20d': round(ret_20d, 4),
            'adj_ret_1d': round(adj_1d, 4),
            'adj_ret_3d': round(adj_3d, 4),
            'adj_ret_5d': round(adj_5d, 4),
            'adj_ret_20d': round(adj_20d, 4),
        })
        
    except Exception as e:
        print(f"Skipping {ticker} on {filing_date}: {e}")

results_df = pd.DataFrame(results)
print(f"Processed {len(results_df)} filings successfully!")
results_df.head()

Processed 450 filings successfully!


,ticker,company_name,accession_no,filed_date,ret_1d,ret_3d,ret_5d,ret_20d,adj_ret_1d,adj_ret_3d,adj_ret_5d,adj_ret_20d
0,AAPL,Apple Inc.,0000320193-23-000104,2023-11-02,-0.0052,0.0239,0.0273,0.0784,-0.0146,0.0099,0.0204,0.0143
1,AAPL,Apple Inc.,0000320193-23-000075,2023-08-03,-0.0480,-0.0595,-0.0690,-0.0159,-0.0427,-0.0589,-0.0617,-0.0172
2,AAPL,Apple Inc.,0001140361-23-023909,2023-05-10,0.0011,-0.0072,-0.0036,0.0418,0.0028,-0.0069,-0.0087,0.0041
3,AAPL,Apple Inc.,0000320193-23-000063,2023-05-04,0.0469,0.0361,0.0480,0.0930,0.0285,0.0218,0.0309,0.0385
4,AAPL,Apple Inc.,0001140361-23-011192,2023-03-10,0.0133,0.0302,0.0438,0.0911,0.0148,0.0224,0.0295,0.0270


In [29]:
# Calculate the mean and standard deviation of day 1 moves
mean_1d = results_df['adj_ret_1d'].abs().mean()
std_1d = results_df['adj_ret_1d'].abs().std()

mean_5d = results_df['adj_ret_5d'].abs().mean()
std_5d = results_df['adj_ret_5d'].abs().std()

print(f"Day 1 average move: {round(mean_1d * 100, 2)}%")
print(f"Day 1 std deviation: {round(std_1d * 100, 2)}%")
print()
print(f"Day 5 average move: {round(mean_5d * 100, 2)}%")
print(f"Day 5 std deviation: {round(std_5d * 100, 2)}%")

Day 1 average move: 1.73%
Day 1 std deviation: 2.24%

Day 5 average move: 2.96%
Day 5 std deviation: 3.1%


In [30]:
# Classify each filing based on market reaction
def classify_reaction(row):
    immediate = abs(row['adj_ret_1d']) >= 0.02  # 2% move on day 1
    delayed = abs(row['adj_ret_1d']) < 0.02 and abs(row['adj_ret_5d']) >= 0.02  # small day 1, big by day 5
    
    if immediate:
        return 'Immediate'
    elif delayed:
        return 'Delayed'
    else:
        return 'No Reaction'

results_df['reaction_type'] = results_df.apply(classify_reaction, axis=1)

# Show the counts
print(results_df['reaction_type'].value_counts())
print()
results_df[['ticker', 'filed_date', 'adj_ret_1d', 'adj_ret_5d', 'reaction_type']].head(10)

reaction_type
No Reaction    173
Delayed        163
Immediate      114
Name: count, dtype: int64



,ticker,filed_date,adj_ret_1d,adj_ret_5d,reaction_type
0,AAPL,2023-11-02,-0.0146,0.0204,Delayed
1,AAPL,2023-08-03,-0.0427,-0.0617,Immediate
2,AAPL,2023-05-10,0.0028,-0.0087,No Reaction
3,AAPL,2023-05-04,0.0285,0.0309,Immediate
4,AAPL,2023-03-10,0.0148,0.0295,Delayed
5,AAPL,2023-02-02,0.0348,0.0238,Immediate
6,MSFT,2023-11-06,0.0084,0.0180,No Reaction
7,MSFT,2023-10-30,-0.0041,0.0092,No Reaction
8,MSFT,2023-10-24,0.0450,0.0356,Immediate
9,MSFT,2023-10-16,-0.0016,0.0258,Delayed


In [31]:

# Step 6: Classify Market Reaction for Each Filing

# We use data-driven thresholds based on the actual average
# moves we calculated from our dataset.
# Thresholds are rounded DOWN to reduce Type 2 errors
# (i.e. we prefer to flag more events rather than miss real ones)

# Threshold for Day 1: if move is less than this, market did NOT react immediately
threshold_1d = 0.015  # 1.5% (rounded down from mean of 1.73%)

# Threshold for Day 5: if move is greater than this, market reacted later (delayed)
threshold_5d = 0.025  # 2.5% (rounded down from mean of 2.96%)

def classify_reaction_final(row):
    # IMMEDIATE: stock moved significantly on day 1 itself
    immediate = abs(row['adj_ret_1d']) >= threshold_1d
    
    # DELAYED: small move on day 1, but big move by day 5
    # This is the key signal we are looking for!
    delayed = abs(row['adj_ret_1d']) < threshold_1d and abs(row['adj_ret_5d']) >= threshold_5d
    
    if immediate:
        return 'Immediate'
    elif delayed:
        return 'Delayed'      # underinterpreted disclosures live here
    else:
        return 'No Reaction'  # market did not react at all

# Apply classification to every filing in our results
results_df['reaction_type'] = results_df.apply(classify_reaction_final, axis=1)

# Print summary of how many filings fall into each category
print(results_df['reaction_type'].value_counts())

reaction_type
Immediate      180
No Reaction    171
Delayed         99
Name: count, dtype: int64


In [32]:
threshold_1d = 0.0173  # exact mean from our data
threshold_5d = 0.0296  # exact mean from our data

def classify_reaction_final(row):
    immediate = abs(row['adj_ret_1d']) >= threshold_1d
    delayed = abs(row['adj_ret_1d']) < threshold_1d and abs(row['adj_ret_5d']) >= threshold_5d
    
    if immediate:
        return 'Immediate'
    elif delayed:
        return 'Delayed'
    else:
        return 'No Reaction'

results_df['reaction_type'] = results_df.apply(classify_reaction_final, axis=1)
print(results_df['reaction_type'].value_counts())

reaction_type
No Reaction    216
Immediate      151
Delayed         83
Name: count, dtype: int64


In [33]:
# Final thresholds - 2% works best for our dataset
# 
threshold_1d = 0.02  # 2%
threshold_5d = 0.02  # 2%

def classify_reaction_final(row):
    immediate = abs(row['adj_ret_1d']) >= threshold_1d
    delayed = abs(row['adj_ret_1d']) < threshold_1d and abs(row['adj_ret_5d']) >= threshold_5d
    
    if immediate:
        return 'Immediate'
    elif delayed:
        return 'Delayed'
    else:
        return 'No Reaction'

results_df['reaction_type'] = results_df.apply(classify_reaction_final, axis=1)
print(results_df['reaction_type'].value_counts())

reaction_type
No Reaction    173
Delayed        163
Immediate      114
Name: count, dtype: int64


In [34]:
# -------------------------------------------------------
# Step 7: Add Placeholder Importance Score
# -------------------------------------------------------
# This column will be filled in by Person 3 (NLP/AI Engineer)
# once they run their LLM classification on the filing text.
# For now we add a placeholder so the table is complete
# and Person 5 can already build the dashboard around it.

# Placeholder value is None - means "not yet scored"
results_df['importance_score'] = None
results_df['importance_label'] = 'Pending'  # High / Medium / Low

print("Importance score column added!")
print(results_df[['ticker', 'filed_date', 'reaction_type', 'importance_score', 'importance_label']].head(10))

Importance score column added!
  ticker filed_date reaction_type importance_score importance_label
0   AAPL 2023-11-02       Delayed             None          Pending
1   AAPL 2023-08-03     Immediate             None          Pending
2   AAPL 2023-05-10   No Reaction             None          Pending
3   AAPL 2023-05-04     Immediate             None          Pending
4   AAPL 2023-03-10       Delayed             None          Pending
5   AAPL 2023-02-02     Immediate             None          Pending
6   MSFT 2023-11-06   No Reaction             None          Pending
7   MSFT 2023-10-30   No Reaction             None          Pending
8   MSFT 2023-10-24     Immediate             None          Pending
9   MSFT 2023-10-16       Delayed             None          Pending


In [35]:
# -------------------------------------------------------
# Step 8: Save Results to CSV
# -------------------------------------------------------
# This is the final output of NB3.
# It will be used by:
# - Person 3 to add importance scores
# - Person 4 (you) to build the watchlist in NB4
# - Person 5 to display on the dashboard

output_path = '/Users/parulchaudhary/Desktop/SEM 2/Predictive/Group Project/market_reactions.csv'

results_df.to_csv(output_path, index=False)

print(f"✅ Saved {len(results_df)} filings to CSV!")
print(f"📁 Location: {output_path}")
print()
print("Columns saved:")
print(results_df.columns.tolist())

✅ Saved 450 filings to CSV!
📁 Location: /Users/parulchaudhary/Desktop/SEM 2/Predictive/Group Project/market_reactions.csv

Columns saved:
['ticker', 'company_name', 'accession_no', 'filed_date', 'ret_1d', 'ret_3d', 'ret_5d', 'ret_20d', 'adj_ret_1d', 'adj_ret_3d', 'adj_ret_5d', 'adj_ret_20d', 'reaction_type', 'importance_score', 'importance_label']
